# Machine Learning - Predicting Future Customer Spending

## Objective

The objective of this notebook is to use historical customer purchasing behaviour to predict how much an existing customer will spend during the following 30 days using Linear Regression.


## Inputs

- Cleaned_Online_Retail_Dataset.csv.
- Python libraries.

## Outputs

- A trained **Linear Regression model** using a Scikit-learn Pipeline.
- Predictions of customer spending for the following 30 days.
- MAE, RMSE and R² model evaluation results.
- An Actual vs Predicted spending visualisation.
- A conclusion explaining the model's prediction performance.

In [1]:
import os

# Change directly to the project root folder
os.chdir('/Users/umarhussain/Documents/Unit3-Data-and-Ai-Ethics')

# Confirm the current working directory
os.getcwd()

'/Users/umarhussain/Documents/Unit3-Data-and-Ai-Ethics'

In [2]:
# Import libraries for data manipulation
import pandas as pd
import numpy as np

# Import libraries for data visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Import train_test_split to create training and testing datasets
from sklearn.model_selection import train_test_split

# Import StandardScaler to scale the model features
from sklearn.preprocessing import StandardScaler

# Import Linear Regression
from sklearn.linear_model import LinearRegression

# Import Pipeline to combine preprocessing and modelling
from sklearn.pipeline import Pipeline

# Import metrics used to evaluate the regression model
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [3]:
# Load the cleaned dataset created during ETL
df = pd.read_csv(
    'Data/Cleaned_Online_Retail_Dataset.csv'
)
# Convert InvoiceDate to datetime format
df['InvoiceDate'] = pd.to_datetime(
    df['InvoiceDate']
)
# Preview the dataset
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TransactionType,TransactionCategory,TotalTransactionValue,Year,Month,YearMonth
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,Sale,Product,15.30,2010,12,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Sale,Product,20.34,2010,12,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,Sale,Product,22.00,2010,12,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Sale,Product,20.34,2010,12,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,Sale,Product,20.34,2010,12,2010-12


In [4]:
# Check the earliest transaction date
print(
    "Earliest transaction:",
    df['InvoiceDate'].min()
)
# Check the latest transaction date
print(
    "Latest transaction:",
    df['InvoiceDate'].max()
)

Earliest transaction: 2010-12-01 08:26:00
Latest transaction: 2011-12-09 12:50:00


## Prepare Data for Machine Learning

The model is designed to predict future customer product spending.

Only completed product sales are therefore included. Returns, cancellations, postage, discounts, fees and other non-product transactions are excluded because they do not represent completed product purchases.

Customers must also have a valid CustomerID so their historical purchases can be connected to their future spending.

In [5]:
# Select product sales
ml_df = df[
    (df['TransactionType'] == 'Sale') &
    (df['TransactionCategory'] == 'Product')
].copy()

# Exclude Customer 15287 
ml_df = ml_df[
    ml_df['CustomerID'].astype(str) != '15287'
].copy()

# Display the number of transactions and customers
print(f"Product sale rows: {len(ml_df):,}")
print(f"Customers: {ml_df['CustomerID'].nunique():,}")



Product sale rows: 391,133
Customers: 4,333


## Define Historical and Prediction Periods

Transactions before **1 November 2011** will be used to describe each customer's historical purchasing behaviour.

Customer purchases between **1 November and 30 November 2011** will then be used to calculate future spending.


In [6]:
# Set the date when the future prediction period begins
cutoff_date = pd.Timestamp(
    '2011-11-01'
)

# Set the end of the 30-day prediction period
prediction_end = (
    cutoff_date +
    pd.Timedelta(days=30)
)

# Display the periods
print(
    "Historical period: before",
    cutoff_date.date()
)

print(
    "Prediction period:",
    cutoff_date.date(),
    "to",
    (
        prediction_end -
        pd.Timedelta(days=1)
    ).date()
)

Historical period: before 2011-11-01
Prediction period: 2011-11-01 to 2011-11-30


## Create Historical Customer Features

Historical transactions are grouped by CustomerID to create one row for each existing customer.

Only transactions before the prediction period are used to calculate these features.

In [7]:
# Select transactions that occurred before the future prediction period
historical_df = ml_df[
    ml_df['InvoiceDate'] < cutoff_date
].copy()

# Confirm that no future transactions have entered the historical dataset
print(
    "Latest historical transaction:",
    historical_df['InvoiceDate'].max()
)

Latest historical transaction: 2011-10-31 17:13:00


In [8]:
# Group historical transactions by customer
customer_features = (
    historical_df
    .groupby('CustomerID')
    .agg(
        LastPurchase=('InvoiceDate', 'max'),
        Frequency=('InvoiceNo', 'nunique'),
        Monetary=('TotalTransactionValue', 'sum')
    )
    .reset_index()
)

# Calculate the number of days since each customer's most recent purchase
customer_features['Recency'] = (
    cutoff_date -
    customer_features['LastPurchase']
).dt.days

# Remove LastPurchase because the model will use the calculated Recency value
customer_features.drop(
    columns='LastPurchase',
    inplace=True
)

# Preview the customer features
customer_features.head()

,CustomerID,Frequency,Monetary,Recency
0,12346,1,77183.60,286
1,12347,6,4085.18,0
2,12348,4,1437.24,36
3,12350,1,294.40,271
4,12352,6,1154.01,33


## Create Future 30-Day Spending

The target variable is `Future30DaySpending`.

It represents how much each existing customer spent on products during the 30 days following the prediction date.

Customers who had historical purchasing behaviour but made no purchases during the following 30 days are assigned a future spending value of £0.

In [9]:
# Select purchases made during the following 30-day prediction period
future_df = ml_df[
    (ml_df['InvoiceDate'] >= cutoff_date) &
    (ml_df['InvoiceDate'] < prediction_end)
].copy()

# Check the future transaction date range
print(
    "First future transaction:",
    future_df['InvoiceDate'].min()
)

print(
    "Last future transaction:",
    future_df['InvoiceDate'].max()
)

First future transaction: 2011-11-01 08:16:00
Last future transaction: 2011-11-30 17:37:00


In [10]:
# Calculate future 30-day spending for each customer
future_spending = (
    future_df
    .groupby('CustomerID')[
        'TotalTransactionValue'
    ]
    .sum()
    .reset_index()
)

# Rename the spending column to clearly identify the machine learning target
future_spending.rename(
    columns={
        'TotalTransactionValue':
        'Future30DaySpending'
    },
    inplace=True
)

# Preview the target values
future_spending.head()

,CustomerID,Future30DaySpending
0,12349,1457.55
1,12352,231.73
2,12356,58.35
3,12357,6207.67
4,12362,417.94


In [11]:
# Join future spending onto the historical customer features
customer_data = customer_features.merge(
    future_spending,
    on='CustomerID',
    how='left'
)

# Customers who did not purchase during the future period are assigned future spending of £0
customer_data['Future30DaySpending'] = (
    customer_data['Future30DaySpending']
    .fillna(0)
)

# Preview the completed modelling dataset
customer_data.head()

,CustomerID,Frequency,Monetary,Recency,Future30DaySpending
0,12346,1,77183.60,286,0.00
1,12347,6,4085.18,0,0.00
2,12348,4,1437.24,36,0.00
3,12350,1,294.40,271,0.00
4,12352,6,1154.01,33,231.73


In [12]:
# Check the number of customers
print(
    f"Customers used for modelling: "
    f"{len(customer_data):,}"
)

# Check for missing values
print("\nMissing values:")
print(customer_data.isnull().sum())

# Confirm Customer 15287 is not present
print(
    "\nCustomer 15287 present:",
    customer_data['CustomerID']
    .astype(str)
    .eq('15287')
    .any()
)

Customers used for modelling: 3,969

Missing values:
CustomerID             0
Frequency              0
Monetary               0
Recency                0
Future30DaySpending    0
dtype: int64

Customer 15287 present: False


The historical customer features were combined with the future spending target.

Customers with no purchases during the future 30-day period were retained with a spending value of £0 rather than being removed, because making no future purchase is a genuine possible customer outcome.

The dataset was also checked for missing values before modelling.

## Select Features and Target

Recency, Frequency and Monetary are used as the independent variables.

Future30DaySpending is used as the dependent variable that the model will predict.

CustomerID is not included as a predictor because it is an identifier rather than a measure of customer behaviour.

In [13]:
# Select the independent variables
X = customer_data[
    [
        'Recency',
        'Frequency',
        'Monetary'
    ]
]

# Select future spending as the target
y = customer_data[
    'Future30DaySpending'
]

# Preview the features
X.head()

,Recency,Frequency,Monetary
0,286,1,77183.60
1,0,6,4085.18
2,36,4,1437.24
3,271,1,294.40
4,33,6,1154.01


## Train/Test Split

The customer-level dataset is split into training and testing data.

80% of customers are used to train the model and 20% are kept as unseen testing data.

The testing data allows the model to be evaluated on customers that were not used during training.

In [14]:
# Split the customer dataset into training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Display the size of each dataset
print(
    "Training customers:",
    len(X_train)
)

print(
    "Testing customers:",
    len(X_test)
)

Training customers: 3175
Testing customers: 794


## Train Linear Regression

A Pipeline is used to combine feature scaling and Linear Regression.

StandardScaler standardises Recency, Frequency and Monetary before the model is trained. It transforms each feature so that it has a mean of approximately 0 and a standard deviation of 1.

This puts the three features onto a similar scale, which is useful because they are measured using very different units. For example, Recency is measured in days, Frequency represents the number of orders, and Monetary represents customer spending in pounds.

Linear Regression is then used because the target, future customer spending, is a continuous numerical value.

Using a Pipeline ensures that the same preprocessing steps are applied consistently to both the training and testing data.

In [15]:
# Create a Pipeline
model_pipeline = Pipeline([
    (
        'scaler',
        StandardScaler()
    ),
    (
        'linear_regression',
        LinearRegression()
    )
])

# Train the Pipeline using the training data
model_pipeline.fit(
    X_train,
    y_train
)

# Generate predictions for the unseen test customers
y_pred = model_pipeline.predict(
    X_test
)

# Display the first ten predictions
y_pred[:10]

array([155.00816485,  15.15779377,  35.97136483, 165.91967347,
        87.40841564,  25.1766551 ,  35.86469775, 242.43603966,
        29.08132232,  59.02556966])

## Evaluate Model

The model is evaluated using three regression metrics:

- **MAE** shows the average absolute difference between predicted and actual customer spending.
- **RMSE** also measures prediction error but gives greater importance to larger errors.
- **R²** shows how much of the variation in future customer spending is explained by the model.

The metrics are calculated using the unseen testing data.

In [16]:
# Calculate MAE
mae = mean_absolute_error(
    y_test,
    y_pred
)

# Calculate RMSE
rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

# Calculate R²
r2 = r2_score(
    y_test,
    y_pred
)

# Display the model evaluation results
print(
    f"MAE: £{mae:,.2f}"
)

print(
    f"RMSE: £{rmse:,.2f}"
)

print(
    f"R²: {r2:.3f}"
)

MAE: £239.01
RMSE: £596.99
R²: 0.450


### Model Evaluation Results

The Linear Regression model achieved:

- **MAE: £239.01**
- **RMSE: £596.99**
- **R²: 0.450**

The MAE means that the model's predicted future spending differed from actual customer spending by approximately **£239.01 per customer on average**.

The RMSE was higher than the MAE, showing that some customers produced larger prediction errors.

The R² score of **0.450** means that approximately **45% of the variation in future 30-day customer spending** was explained by the Recency, Frequency and Monetary features. The remaining variation is likely influenced by other factors that are not included in the model.

Overall, the results indicate that historical customer purchasing behaviour provides **moderate predictive ability** for estimating future customer spending. While the model does not explain all differences in future spending, it demonstrates that previous purchasing behaviour can provide useful information when predicting future customer value.

In [17]:
# Create a dataframe containing actual and predicted customer spending
results = pd.DataFrame({
    'ActualSpending': y_test.values,
    'PredictedSpending': y_pred
})

# Round spending values to 2 decimal places
results = results.round(2)

# Preview the results
results.head(20)

,ActualSpending,PredictedSpending
0,0.00,155.01
1,0.00,15.16
2,0.00,35.97
3,0.00,165.92
4,0.00,87.41
5,0.00,25.18
6,0.00,35.86
7,187.48,242.44
8,0.00,29.08
9,0.00,59.03


In [18]:
# Create the figure
plt.figure(figsize=(8, 6))

# Plot spending against predicted spending
sns.scatterplot(
    data=results,
    x='ActualSpending',
    y='PredictedSpending',
    alpha=0.6
)

# Find the minimum and maximum values
minimum_value = min(
    results['ActualSpending'].min(),
    results['PredictedSpending'].min()
)

maximum_value = max(
    results['ActualSpending'].max(),
    results['PredictedSpending'].max()
)

# Add a dashed line showing where predictions would appear
plt.plot(
    [minimum_value, maximum_value],
    [minimum_value, maximum_value],
    linestyle='--'
)

# Add chart labels
plt.title(
    'Actual vs Predicted Future Customer Spending'
)

plt.xlabel(
    'Actual Future 30-Day Spending (£)'
)

plt.ylabel(
    'Predicted Future 30-Day Spending (£)'
)

# Display the graph
plt.show()

<Figure size 800x600 with 1 Axes>

The scatterplot compares the actual future spending of each test customer with the amount predicted by the Linear Regression model.

The dashed diagonal line represents a perfect prediction, where the predicted spending is equal to the actual spending. Points closer to this line indicate more accurate predictions, while points further away represent larger prediction errors. Points above the line show over-predictions, while points below the line show under-predictions.

In [19]:

# Import Path to create and manage file and folder paths
from pathlib import Path

# Define the folder where Version 1 model outputs will be saved
output_folder = Path("Output/v1")

# Create the folder if it does not already exist
output_folder.mkdir(parents=True, exist_ok=True)

# Save the actual and predicted customer spending values to a CSV file
results.to_csv(
    output_folder / "model_predictions.csv",
    index=False
)

# Display a message showing where the file was saved
print(
    "Predictions saved successfully to:",
    (output_folder / "model_predictions.csv").resolve()
)


Predictions saved successfully to: /Users/umarhussain/Documents/Unit3-Data-and-Ai-Ethics/Output/v1/model_predictions.csv
